# Trích xuất ASR từ video trên Kaggle
Chạy các cell theo thứ tự, bật **Internet** và **GPU**. Notebook clone nhánh `vbs-dev`, chọn dải video ID V3C, tải tối đa 10 video `<video_id>.mp4` mỗi lượt từ Drive, gọi pipeline ASR của repo và upload **chỉ** các file `<video_id>.json` lên Drive đích bằng một lệnh rclone cho mỗi lượt. Lô cuối có dưới 10 video vẫn được upload.

Faster-Whisper được cài vào Python hiện tại của Kaggle một lần, không tạo venv và không cài toàn bộ `requirements.txt`. Model `Systran/faster-whisper-large-v3` dùng INT8 và được cache cho các lượt tiếp theo trong cùng phiên. Đường dẫn `rclone.conf` phải trỏ tới Kaggle Input của bạn.

In [ ]:
# 1. Cấu hình
import importlib.metadata, json, os, re, shutil, subprocess, sys, urllib.request, zipfile
from pathlib import Path
REPO_URL = 'https://github.com/iamtro-2006/event-retrieval-system.git'
REPO_BRANCH = 'vbs-dev'  # push code ASR và notebook lên nhánh này trước khi chạy
DATASET = 'V3C'
START_VIDEO_ID = 1
END_VIDEO_ID = 10
VIDEO_EXTENSION = '.mp4'
SOURCE_ROOT_FOLDER_ID = '19b7skL2b0PRJWWweWFh8qVNTLQDF6x9K'
DEST_ROOT_FOLDER_ID = '1mr6OrhXbXpstdZhXnkpkqPWiOHkEow8j'
RCLONE_CONFIG_SOURCE = Path('/kaggle/input/datasets/vnnoname/rclone-new/rclone.conf')
RCLONE_REMOTE = 'iamtro'
UPLOAD_BATCH_SIZE = 10
OVERWRITE_REMOTE = False

def video_id(number):
    return f'{number:05d}'

if DATASET != 'V3C' or not 1 <= START_VIDEO_ID <= END_VIDEO_ID <= 99999:
    raise ValueError('Cần 1 <= START_VIDEO_ID <= END_VIDEO_ID <= 99999 cho V3C')
if SOURCE_ROOT_FOLDER_ID == DEST_ROOT_FOLDER_ID:
    raise ValueError('Folder Drive nguồn và đích phải khác nhau')
if not RCLONE_CONFIG_SOURCE.is_file():
    raise FileNotFoundError(f'Chưa gắn rclone config: {RCLONE_CONFIG_SOURCE}')
WORK = Path('/kaggle/working/asr_pipeline')
WORK.mkdir(parents=True, exist_ok=True)
JOBS_ROOT = WORK / 'jobs'
JOBS_ROOT.mkdir(exist_ok=True)
STAGE_DIR = WORK / 'pending_json'
STAGE_DIR.mkdir(exist_ok=True)
RCLONE_CONFIG = WORK / 'rclone.conf'
shutil.copyfile(RCLONE_CONFIG_SOURCE, RCLONE_CONFIG)
RCLONE_CONFIG.chmod(0o600)
REPO_DIR = WORK / 'event-retrieval-system'
BACKEND_DIR = REPO_DIR / 'backend'
print('Workspace:', WORK)


In [ ]:
# 2. Clone/pull repo, cài dependency tối thiểu và chuẩn bị rclone
def run(args, *, cwd=None, env=None):
    args = [str(item) for item in args]
    print('+', ' '.join(args[:5]), '...' if len(args) > 5 else '', flush=True)
    return subprocess.run(args, cwd=cwd, env=env, check=True, text=True)

if not REPO_DIR.exists():
    run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', REPO_BRANCH, REPO_URL, REPO_DIR])
else:
    branch = subprocess.check_output(['git', '-C', str(REPO_DIR), 'branch', '--show-current'], text=True).strip()
    if branch != REPO_BRANCH:
        raise RuntimeError(f'Checkout đang ở {branch}, cần {REPO_BRANCH}')
    run(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if not (BACKEND_DIR / 'src/asr_extraction/pipeline/extract_asr.py').is_file():
    raise RuntimeError('Nhánh GitHub chưa có ASR extraction; hãy push code lên vbs-dev')
print('ASR commit:', commit)

try:
    fw_version = importlib.metadata.version('faster-whisper')
except importlib.metadata.PackageNotFoundError:
    fw_version = None
if fw_version != '1.2.1':
    run([sys.executable, '-m', 'pip', 'install', '--upgrade-strategy', 'only-if-needed', 'faster-whisper==1.2.1'])
try:
    import yaml
except ImportError:
    run([sys.executable, '-m', 'pip', 'install', 'PyYAML'])
    import yaml

rclone = shutil.which('rclone')
if rclone is None and shutil.which('apt-get'):
    try:
        run(['apt-get', 'update', '-qq'])
        run(['apt-get', 'install', '-y', '-qq', 'rclone'])
        rclone = shutil.which('rclone')
    except subprocess.CalledProcessError:
        pass
if rclone is None:
    release = 'v1.75.2'
    archive = WORK / f'rclone-{release}-linux-amd64.zip'
    url = f'https://github.com/rclone/rclone/releases/download/{release}/{archive.name}'
    with urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'kaggle-asr'}), timeout=120) as src, archive.open('wb') as dst:
        shutil.copyfileobj(src, dst)
    with zipfile.ZipFile(archive) as zf:
        matches = [item for item in zf.infolist() if item.filename.endswith('/rclone') and not item.is_dir()]
        if len(matches) != 1:
            raise RuntimeError('Không tìm thấy binary rclone trong ZIP release')
        rclone = WORK / 'rclone'
        with zf.open(matches[0]) as src, rclone.open('wb') as dst:
            shutil.copyfileobj(src, dst)
    rclone.chmod(0o755)
run([rclone, 'version'])


In [ ]:
# 3. Liệt kê video nguồn và bỏ qua JSON đã upload
def remote_args(folder_id):
    return [rclone, '--config', RCLONE_CONFIG, '--drive-root-folder-id', folder_id]

source_names = set(subprocess.check_output(
    [*map(str, remote_args(SOURCE_ROOT_FOLDER_ID)), 'lsf', f'{RCLONE_REMOTE}:', '--files-only'],
    text=True).splitlines())
selected = [video_id(number) for number in range(START_VIDEO_ID, END_VIDEO_ID + 1)]
missing = [vid for vid in selected if f'{vid}{VIDEO_EXTENSION}' not in source_names]
if missing:
    raise FileNotFoundError(f'Thiếu video trong Drive nguồn: {missing}')
dest_names = set(subprocess.check_output(
    [*map(str, remote_args(DEST_ROOT_FOLDER_ID)), 'lsf', f'{RCLONE_REMOTE}:', '--files-only'],
    text=True).splitlines())
pending = [vid for vid in selected if OVERWRITE_REMOTE or f'{vid}.json' not in dest_names]
print(f'Đã chọn {len(selected)} video; cần xử lý {len(pending)}:', pending)


In [ ]:
# 4. Mỗi lô tải tối đa 10 video, chạy ASR một lần, upload JSON rồi dọn tạm
cfg = yaml.safe_load((BACKEND_DIR / 'configs/asr_extraction.yaml').read_text(encoding='utf-8'))
model_cfg = cfg['extraction']
if (model_cfg['model'] != 'Systran/faster-whisper-large-v3' or
        model_cfg['compute_type'] != 'int8' or model_cfg.get('language') is not None):
    raise RuntimeError('ASR config trên nhánh này chưa dùng multilingual large-v3 INT8')
ASR_ENV = os.environ.copy()
ASR_ENV['PYTHONPATH'] = str(BACKEND_DIR)
ASR_ENV['HF_HOME'] = str(WORK / 'hf_cache')  # cache model dùng lại qua các lô
cuda_libs = []
for module_name in ('nvidia.cublas', 'nvidia.cudnn'):
    try:
        module = __import__(module_name, fromlist=['__path__'])
        library = Path(module.__path__[0]) / 'lib'
        if library.is_dir():
            cuda_libs.append(str(library))
    except ImportError:
        pass
if cuda_libs:
    ASR_ENV['LD_LIBRARY_PATH'] = ':'.join(cuda_libs + [os.environ.get('LD_LIBRARY_PATH', '')])

def validate_json(path, vid):
    data = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(data, list):
        raise ValueError(f'{path}: cần list segment')
    for item in data:
        if (not isinstance(item, dict) or item.get('video_id') != vid or
                not isinstance(item.get('transcript'), str) or
                not isinstance(item.get('start'), (int, float)) or
                not isinstance(item.get('end'), (int, float)) or
                not 0 <= item['start'] < item['end']):
            raise ValueError(f'{path}: segment không đúng schema')
    return len(data)

completed = {}
for start in range(0, len(pending), UPLOAD_BATCH_SIZE):
    batch_ids = pending[start:start + UPLOAD_BATCH_SIZE]
    batch_work = JOBS_ROOT / f'batch_{start // UPLOAD_BATCH_SIZE:04d}'
    if batch_work.exists() or list(STAGE_DIR.glob('*.json')):
        raise RuntimeError(f'Còn dữ liệu từ lượt trước tại {batch_work} hoặc {STAGE_DIR}; kiểm tra trước khi chạy lại')
    video_dir = batch_work / 'videos' / DATASET
    output_dir = batch_work / 'asr' / DATASET
    video_dir.mkdir(parents=True)
    for vid in batch_ids:
        path = video_dir / f'{vid}{VIDEO_EXTENSION}'
        run([*remote_args(SOURCE_ROOT_FOLDER_ID), 'copyto',
             f'{RCLONE_REMOTE}:{path.name}', path])
        if not path.is_file() or path.stat().st_size == 0:
            raise RuntimeError(f'Video tải về rỗng: {path.name}')
    run([sys.executable, '-m', 'scripts.asr.extract.run', '--dataset', DATASET.lower(),
         '--input-dir', video_dir, '--output-dir', output_dir],
        cwd=BACKEND_DIR, env=ASR_ENV)
    for vid in batch_ids:
        result = output_dir / f'{vid}.json'
        if not result.is_file():
            raise FileNotFoundError(f'Không có ASR JSON: {result}')
        completed[vid] = validate_json(result, vid)
        shutil.move(result, STAGE_DIR / result.name)
        print(f'ASR xong {vid}: {completed[vid]} đoạn')
    for path in video_dir.iterdir():
        path.unlink()  # JSON đã được xác nhận; giải phóng dung lượng trước upload
    run([*remote_args(DEST_ROOT_FOLDER_ID), 'copy', STAGE_DIR, f'{RCLONE_REMOTE}:',
         '--transfers', str(min(UPLOAD_BATCH_SIZE, len(batch_ids))), '--checkers', '10'])
    run([*remote_args(DEST_ROOT_FOLDER_ID), 'check', STAGE_DIR, f'{RCLONE_REMOTE}:', '--one-way'])
    for path in STAGE_DIR.glob('*.json'):
        path.unlink()
    shutil.rmtree(batch_work)
    print(f'Đã upload {len(batch_ids)} JSON và dọn video tạm')
print('Hoàn tất:', completed, '| commit:', commit)
